In [ ]:
from typing import Tuple
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F

device = ""
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

def load_data(p: Path | str, train_data_len, val_data_len) -> Tuple[bytes, bytes]:
    with open(p, "r") as fp:
        text = fp.read()
    train_data = text[:train_data_len]
    val_data = text[train_data_len:train_data_len+val_data_len]
    return train_data.encode("utf-8"), val_data.encode("utf-8")

train_data, val_data = load_data("tiny-shakespeare.txt", 1000000, 100000)


In [ ]:
from collections import Counter

class BPETokenizer:
    def __init__(self):
        self.vocab = [bytes([i]) for i in range(256)]
        self.merges = {}

    def merge_all(self, ids: list[int], pair, new_id):
        i = 0
        res = []
        while i < len(ids):
            if i+1 < len(ids) and (ids[i], ids[i+1]) == pair:
                res.append(new_id)
                i += 2
            else:
                res.append(ids[i])
                i += 1
        return res

    def train(self, data: bytes, vocab_size: int):
        ids = list(data)
        print(len(ids))
        while len(self.vocab) < vocab_size and len(ids) > 1:
            stats = Counter(zip(ids, ids[1:]))
            pair = max(stats, key=lambda p: stats[p])
            new_id = len(self.vocab)
            self.merges[pair] = new_id
            self.vocab.append(self.vocab[pair[0]]+self.vocab[pair[1]])
            ids = self.merge_all(ids, pair, new_id)

    def encode(self, data: bytes):
        ids = list(data)
        while len(ids) > 1:
            pairs = set(zip(ids, ids[1:]))
            p = min(pairs, key=lambda p: self.merges.get(p, 10**9))
            if p not in self.merges:
                break
            ids = self.merge_all(ids, p, self.merges[p])
        return ids

    def decode(self, ids: list[int]):
        return b"".join(map(lambda id: self.vocab[id], ids))

tokenizer = BPETokenizer()
tokenizer.train(train_data, 500)
# print(tokenizer.vocab)
# print(tokenizer.encode(b"for"))
# print(tokenizer.decode([299, 299]))

In [ ]:
class Attention(nn.Module):
    def __init__(self, d_model, context_len):
        super().__init__()
        self.d_model = d_model
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)
        self.register_buffer("mask", torch.tril(torch.ones((context_len, context_len), dtype=torch.bool)))

    def forward(self, x: torch.Tensor):
        batch_size, seq_length, d_model = x.shape
        K = self.Wk(x)
        Q = self.Wq(x)
        V = self.Wv(x)
        scores = Q @ K.transpose(-1, -2) / (self.d_model**0.5)
        scores = scores.masked_fill(self.mask[:seq_length, :seq_length] == 0, float("-inf"))
        scores = torch.softmax(scores, dim=-1)
        output = scores @ V
        return output

class MLP(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.up_proj = nn.Linear(d_model, 4 * d_model, bias=False)
        self.down_proj = nn.Linear(4 * d_model, d_model, bias=False)

    def forward(self, x: torch.Tensor):
        return self.down_proj(F.gelu(self.up_proj(x)))

class AttentionBlock(nn.Module):
    def __init__(self, d_model: int, context_len: int):
        super().__init__()
        self.attn = Attention(d_model, context_len)
        self.ln1 = nn.LayerNorm(d_model)
        self.ffn = MLP(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor):
        x = x+self.attn(self.ln1(x))
        x = x+self.ffn(self.ln2(x))
        return x
